### Motivation

In this notebook, we'll calculate several indicator to give the sentiment of the market and signal or alert to help manage expected risk

### Methodology

#### VIX

CBOE vix is the average expected volatility over next 30 days

##### calculation

derive expected volatility by averaging the weighted prices of out of the money puts and calls

### Code

In [78]:
# import packages
import yfinance as yf
import pandas as pd
import numpy as np

### Risk ON/OFF Indicator

In [56]:
# configuration
risk_on_down_instruments = ["VIXY", "GLD", "BND", "IDU", [["SPLV", "SPHB"], "/"]]
risk_on_up_instruments = [
    "IYT",
    "COPX",
    "QQQ",
    [["JPY=X", "NZD=X"], "/"],
    "DBB",
    "HYG",
    "EEMO",
    [["FNDA", "SCHX"], "/"],
    [["FDIS", "FXG"], "/"],
]

In [108]:
# helper function
def get_indicator(tickers):
    if not isinstance(tickers, list):
        stock = yf.Ticker(tickers)
        change_pct = ((stock.info["ask"] + stock.info["bid"]) / 2.0 / stock.info["previousClose"] - 1) * 100
        return {"change_pct": change_pct, "ticker": tickers}
    else:
        ticker1, ticker2 = tickers[0]
        stock1 = yf.Ticker(ticker1)
        stock2 = yf.Ticker(ticker2)

        op = tickers[1]
        if op == "/":
            stock1_mid = (stock1.info["bid"] + stock1.info["ask"]) / 2.0
            stock2_mid = (stock2.info["bid"] + stock2.info["ask"]) / 2.0
            stock1_prev_close = stock1.info["previousClose"]
            stock2_prev_close = stock2.info["previousClose"]
            change_pct = ((stock1_mid / stock2_mid) / (stock1_prev_close / stock2_prev_close) - 1) * 100
            return {"change_pct": change_pct, "ticker": "/".join(tickers[0])}
        else:
            raise Exception("unknown operation %s" % (op))


def color_negative_red(val):
    if isinstance(val, float):
        color = "red" if val < 0 else "green"
        return "color: %s" % color
    else:
        return ""

In [92]:
# indicaotr calculation
risk_on_up_ticker_indicators = []
for tickers in risk_on_up_instruments:
    indicator = get_indicator(tickers)
    risk_on_up_ticker_indicators.append(indicator)

risk_on_down_ticker_indicators = []
for tickers in risk_on_down_instruments:
    indicator = get_indicator(tickers)
    risk_on_down_ticker_indicators.append(indicator)

risk_on_up_ticker_indicators = pd.DataFrame(risk_on_up_ticker_indicators)
risk_on_up_ticker_indicators.loc[:, "risk_mv"] = "up"

risk_on_down_ticker_indicators = pd.DataFrame(risk_on_down_ticker_indicators)
risk_on_down_ticker_indicators.loc[:, "risk_mv"] = "down"

risk_on_indicator_df = pd.concat([risk_on_down_ticker_indicators, risk_on_up_ticker_indicators])
risk_on_indicator_df = risk_on_indicator_df.reset_index(drop=True)

#### Results

In [97]:
risk_on_indicator_df = risk_on_indicator_df[["risk_mv", "ticker", "change_pct"]].sort_values(["risk_mv", "change_pct"])

In [120]:
np.round(risk_on_indicator_df, 2).set_index(["risk_mv", "ticker"]).style.applymap(color_negative_red).set_table_styles(
    [
        {
            "selector": "th",
            "props": [
                ("font-size", "10pt"),
                ("border-style", "solid"),
                ("border-width", "1px"),
            ],
        }
    ]
)